
# Kickstarter — Proposed Method 2
## GBDT Leaf Embedding + MLP + Retrieval

This notebook extends the existing GBDT leaf-embedding + MLP pipeline by adding an
iLTM-inspired retrieval stage.

**Important:** this is not a full reproduction of iLTM. It reproduces the relevant
idea for this experiment: GBDT leaf embeddings → concatenation → fixed-size
representation → MLP + retrieval. The official iLTM repository exposes retrieval
controls such as `do_retrieval`, `retrieval_alpha`, `retrieval_temperature`, and
`retrieval_distance`.


In [1]:

# ============================================================
# 1. IMPORTS
# ============================================================

import json
import time
import joblib
import numpy as np
import pandas as pd

from pathlib import Path

from scipy.sparse import csr_matrix, hstack, issparse

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.neural_network import MLPRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

SEED = 42
np.random.seed(SEED)

print("Imports completed.")


Imports completed.



### Cell 1 — Explanation

Imports the libraries needed for the complete Method 2 pipeline:

**data → preprocessing → XGBoost leaf embedding → PCA → MLP → retrieval → evaluation → saving**.


In [2]:

# ============================================================
# 2. PATHS + METHOD 2 SETTINGS
# ============================================================

TRAIN_FILE = Path(
    r"E:\NSU\cse445\EDA attempt3\Dataset\ML dataset\ML_train.csv"
)

TEST_FILE = Path(
    r"E:\NSU\cse445\EDA attempt3\Dataset\ML dataset\ML_test.csv"
)

# Saved by your previous tuned-XGBoost/GBDT notebook.
XGB_MODEL_FILE = Path(
    r"E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\GBDT proposed 1\FINAL_all_model_components.pkl"
)

OUTPUT_DIR = Path(
    r"E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\GBDT_prop_2"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Same representation sizes used by your current GBDT+MLP pipeline.
RANDOM_DIM = 512
MAIN_DIM = 128

# Retrieval experiment settings.
# These are explicit experiment settings, NOT claimed to be
# the exact hidden defaults of the paper implementation.
RETRIEVAL_K = 32
RETRIEVAL_TEMPERATURE = 0.10
RETRIEVAL_ALPHA = 0.30

print("Output directory:", OUTPUT_DIR)
print("K:", RETRIEVAL_K)
print("Temperature:", RETRIEVAL_TEMPERATURE)
print("Alpha:", RETRIEVAL_ALPHA)


Output directory: E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\GBDT_prop_2
K: 32
Temperature: 0.1
Alpha: 0.3



### Cell 2 — Explanation

The retrieval parameters mean:

- **K = 32:** retrieve the 32 most similar training campaigns.
- **Temperature = 0.10:** controls how sharply similarity becomes a retrieval weight.
- **Alpha = 0.30:** final prediction uses 70% MLP + 30% retrieval.

We will explicitly use:

\[
\hat y_{final}=(1-\alpha)\hat y_{MLP}+\alpha\hat y_{retrieval}
\]

The official iLTM repository exposes the same retrieval concepts, but these numerical
settings are our controlled experiment rather than claimed paper defaults.


In [3]:

# ============================================================
# 3. LOAD OFFICIAL TRAIN/TEST DATA
# ============================================================

if not TRAIN_FILE.exists():
    raise FileNotFoundError(f"Train file not found: {TRAIN_FILE}")

if not TEST_FILE.exists():
    raise FileNotFoundError(f"Test file not found: {TEST_FILE}")

train = pd.read_csv(TRAIN_FILE)
test = pd.read_csv(TEST_FILE)

print("Train shape:", train.shape)
print("Test shape :", test.shape)
display(train.head())


Train shape: (16000, 81)
Test shape : (4000, 81)


,id,goal_usd,log_goal_usd,duration_days,prelaunch_days,name_char_length,name_word_count,blurb_char_length,blurb_word_count,launch_year,...,location_type_County,location_type_Island,location_type_LocalAdmin,location_type_Miscellaneous,location_type_Suburb,location_type_Town,location_type_Unknown,location_type_Zip,target_usd,log_target
0,160053502,1490.1241,7.307286,30.000000,13.537003,30.0,5.0,25.0,5.0,2020.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,2694.360622,7.899287
1,1797462698,5000.0000,8.517393,35.041668,18.665070,8.0,2.0,108.0,18.0,2017.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,122.000000,4.812184
2,1582340481,10000.0000,9.210441,30.000000,10.058495,53.0,7.0,108.0,13.0,2025.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.000000,0.000000
3,498571554,15475.7350,9.647093,29.958334,7.044641,39.0,7.0,110.0,18.0,2022.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,26102.038065,10.169807
4,1914908476,1400.0000,7.244942,60.000000,13.044236,29.0,5.0,125.0,24.0,2025.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1478.000000,7.299121



### Cell 3 — Explanation

Loads the same official encoded train/test files used in your previous experiments.
The official test set remains untouched for fitting and retrieval candidates.


In [4]:

# ============================================================
# 4. TARGETS + LEAKAGE-SAFE FEATURES
# ============================================================

TARGET = "target_usd"
LOG_TARGET = "log_target"
ID_COL = "id"

assert TARGET in train.columns
assert LOG_TARGET in train.columns
assert TARGET in test.columns
assert LOG_TARGET in test.columns

EXCLUDE_COLS = [ID_COL, TARGET, LOG_TARGET]

feature_cols = [
    c for c in train.columns
    if c not in EXCLUDE_COLS
]

assert TARGET not in feature_cols
assert LOG_TARGET not in feature_cols
assert ID_COL not in feature_cols

X_train = train[feature_cols].copy()
X_test = test[feature_cols].copy()

y_train = train[LOG_TARGET].to_numpy()
y_test = test[LOG_TARGET].to_numpy()

actual_usd = test[TARGET].to_numpy()

print("Input features:", len(feature_cols))
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)


Input features: 78
X_train: (16000, 78)
X_test : (4000, 78)



### Cell 4 — Explanation

The target is the log-transformed funding amount.

Target:

\[
y=\log(1+USD)
\]

The raw target and ID are explicitly removed from the feature matrix to prevent leakage.


In [5]:

# ============================================================
# 5. NUMERIC CHECK
# ============================================================

non_numeric = X_train.select_dtypes(
    exclude=[np.number]
).columns.tolist()

if non_numeric:
    raise TypeError(
        "Non-numeric model inputs found: "
        + str(non_numeric)
    )

print("All model inputs are numeric.")


All model inputs are numeric.



### Cell 5 — Explanation

Your ML train/test files are already encoded, so this notebook does not encode
categorical columns again.


In [6]:

# ============================================================
# 6. ROBUST PREPROCESSING
# ============================================================

robust_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", RobustScaler())
])

X_train_pre = robust_preprocessor.fit_transform(X_train)
X_test_pre = robust_preprocessor.transform(X_test)

if issparse(X_train_pre):
    X_train_pre = X_train_pre.toarray()
    X_test_pre = X_test_pre.toarray()

def smooth_clip(X, limit=5.0):
    return np.tanh(X / limit) * limit

X_train_pre = smooth_clip(X_train_pre)
X_test_pre = smooth_clip(X_test_pre)

print("Preprocessed train:", X_train_pre.shape)
print("Preprocessed test :", X_test_pre.shape)


Preprocessed train: (16000, 78)
Preprocessed test : (4000, 78)



### Cell 6 — Explanation

Same preprocessing idea as your existing GBDT+MLP notebook:

**median imputation → RobustScaler → bounded tanh clipping**.

The preprocessor is fitted only on training data.


In [11]:
# ============================================================
# 7. LOAD EXISTING TUNED XGBOOST
# ============================================================

if not XGB_MODEL_FILE.exists():
    raise FileNotFoundError(
        f"Saved tuned XGBoost not found: {XGB_MODEL_FILE}"
    )

xgb_bundle = joblib.load(XGB_MODEL_FILE)

print("Loaded object type:", type(xgb_bundle))
print("Bundle keys:", xgb_bundle.keys())

xgb_model = xgb_bundle["xgb_model"]

print("XGBoost model type:", type(xgb_model))

Loaded object type: <class 'dict'>
Bundle keys: dict_keys(['xgb_model', 'preprocessor', 'leaf_encoder', 'pca', 'final_scaler', 'mlp', 'metadata'])
XGBoost model type: <class 'xgboost.sklearn.XGBRegressor'>



### Cell 7 — Explanation

We intentionally **do not retune XGBoost again**.

This isolates the experiment:

**existing tuned XGBoost representation + retrieval**

rather than changing XGBoost and retrieval simultaneously.


In [12]:

# ============================================================
# 8. TUNED XGBOOST BENCHMARK
# ============================================================

xgb_pred_log = xgb_model.predict(X_test)
xgb_pred_usd = np.maximum(np.expm1(xgb_pred_log), 0.0)

print("XGBoost benchmark predictions generated.")


XGBoost benchmark predictions generated.



### Cell 8 — Explanation

This provides the standalone tuned-XGBoost benchmark against which Method 2 will be
compared.


In [13]:

# ============================================================
# 9. EXTRACT XGBOOST LEAF INDICES
# ============================================================

leaf_train = xgb_model.apply(X_train)
leaf_test = xgb_model.apply(X_test)

print("Leaf train shape:", leaf_train.shape)
print("Leaf test shape :", leaf_test.shape)
print("Example:")
print(leaf_train[:3])


Leaf train shape: (16000, 1200)
Leaf test shape : (4000, 1200)
Example:
[[217. 221. 217. ... 212. 155. 125.]
 [223. 227. 225. ... 133. 149. 125.]
 [222. 226. 224. ... 226. 149. 125.]]



### Cell 9 — Explanation

Each sample is represented by the leaf it reaches in every XGBoost tree.

This is the central GBDT embedding idea used by iLTM's tree-embedding component.


In [14]:

# ============================================================
# 10. ONE-HOT ENCODE LEAF INDICES
# ============================================================

try:
    leaf_encoder = OneHotEncoder(
        sparse_output=True,
        handle_unknown="ignore",
        dtype=np.int8
    )
except TypeError:
    leaf_encoder = OneHotEncoder(
        sparse=True,
        handle_unknown="ignore",
        dtype=np.int8
    )

leaf_train_encoded = leaf_encoder.fit_transform(leaf_train)
leaf_test_encoded = leaf_encoder.transform(leaf_test)

print("Leaf embedding train:", leaf_train_encoded.shape)
print("Leaf embedding test :", leaf_test_encoded.shape)


Leaf embedding train: (16000, 99219)
Leaf embedding test : (4000, 99219)



### Cell 10 — Explanation

Leaf IDs are converted into sparse one-hot features.

This changes:

`tree → leaf number`

into a learned sparse representation that can be concatenated with the original
preprocessed features.


In [15]:

# ============================================================
# 11. CONCATENATE ORIGINAL FEATURES + GBDT EMBEDDING
# ============================================================

X_train_combined = hstack([
    csr_matrix(X_train_pre),
    leaf_train_encoded
]).tocsr()

X_test_combined = hstack([
    csr_matrix(X_test_pre),
    leaf_test_encoded
]).tocsr()

print("Combined train:", X_train_combined.shape)
print("Combined test :", X_test_combined.shape)


Combined train: (16000, 99297)
Combined test : (4000, 99297)



### Cell 11 — Explanation

This reproduces the important `xgbrconcat` idea:

\[
X_{combined}=[X_{original};\Gamma(X)]
\]

where \(\Gamma(X)\) is the GBDT leaf embedding.


In [16]:

# ============================================================
# 12. RANDOM FEATURE EXPANSION + ReLU
# ============================================================

rng = np.random.default_rng(SEED)

combined_dim = X_train_combined.shape[1]

Omega = rng.normal(
    0.0,
    1.0 / np.sqrt(combined_dim),
    size=(combined_dim, RANDOM_DIM)
)

def relu(X):
    return np.maximum(0.0, X)

X_train_random = relu(
    X_train_combined @ Omega
)

X_test_random = relu(
    X_test_combined @ Omega
)

print("Random train:", X_train_random.shape)
print("Random test :", X_test_random.shape)


Random train: (16000, 512)
Random test : (4000, 512)



### Cell 12 — Explanation

The high-dimensional combined representation is projected into a manageable
random-feature space and passed through ReLU.

This follows the structure already used in your current GBDT notebook and is
conceptually aligned with the paper's dimensionality-agnostic representation stage.


In [17]:

# ============================================================
# 13. PCA FIXED-SIZE REPRESENTATION
# ============================================================

effective_main_dim = min(
    MAIN_DIM,
    X_train_random.shape[0],
    X_train_random.shape[1]
)

pca = PCA(
    n_components=effective_main_dim,
    random_state=SEED
)

X_train_emb = pca.fit_transform(X_train_random)
X_test_emb = pca.transform(X_test_random)

print("Embedding train:", X_train_emb.shape)
print("Embedding test :", X_test_emb.shape)
print(
    "Explained variance:",
    pca.explained_variance_ratio_.sum()
)


Embedding train: (16000, 128)
Embedding test : (4000, 128)
Explained variance: 0.6220653188284715



### Cell 13 — Explanation

PCA converts the large random representation into a fixed-size embedding.

Your previous notebook used a 128-dimensional main representation.


In [18]:

# ============================================================
# 14. FINAL STANDARDIZATION
# ============================================================

final_scaler = StandardScaler()

X_train_final = final_scaler.fit_transform(X_train_emb)
X_test_final = final_scaler.transform(X_test_emb)

print("Final train:", X_train_final.shape)
print("Final test :", X_test_final.shape)


Final train: (16000, 128)
Final test : (4000, 128)



### Cell 14 — Explanation

The fixed-size embedding is standardized before it is used by the MLP and retrieval
system.


In [19]:

# ============================================================
# 15. TRAIN MLP
# ============================================================

mlp = MLPRegressor(
    hidden_layer_sizes=(256, 256, 128),
    activation="relu",
    solver="adam",
    learning_rate_init=0.001,
    batch_size=256,
    max_iter=100,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=SEED
)

mlp_start = time.time()
mlp.fit(X_train_final, y_train)
mlp_training_time = time.time() - mlp_start

print(f"MLP training time: {mlp_training_time:.2f} s")
print("Iterations:", mlp.n_iter_)


MLP training time: 11.26 s
Iterations: 16



### Cell 15 — Explanation

The MLP learns the main predictive function from the GBDT-derived representation:

\[
X_{embedding}\rightarrow\hat y_{MLP}
\]

The architecture is kept the same as your current GBDT+MLP experiment.


In [20]:

# ============================================================
# 16. MLP PREDICTIONS
# ============================================================

mlp_pred_log = mlp.predict(X_test_final)
mlp_pred_usd = np.maximum(
    np.expm1(mlp_pred_log),
    0.0
)

print("MLP predictions generated.")


MLP predictions generated.



### Cell 16 — Explanation

Predictions remain in log space during model inference and are converted back to USD
with `expm1` for the real-world funding metrics.


In [21]:

# ============================================================
# 17. BUILD RETRIEVAL MEMORY FROM TRAINING DATA ONLY
# ============================================================

retrieval_index = NearestNeighbors(
    n_neighbors=min(RETRIEVAL_K, len(X_train_final)),
    metric="cosine"
)

retrieval_index.fit(X_train_final)

print(
    "Retrieval candidates:",
    len(X_train_final),
    "training campaigns"
)


Retrieval candidates: 16000 training campaigns



### Cell 17 — Explanation

This is the retrieval memory.

**Only training campaigns are stored as candidates.**

The test target values are never used as retrieval candidates, preventing target
leakage.


In [22]:

# ============================================================
# 18. SOFT k-NN RETRIEVAL
# ============================================================

def soft_retrieval_predict(
    query_matrix,
    candidate_matrix,
    candidate_targets_log,
    k,
    temperature
):
    k = min(k, candidate_matrix.shape[0])

    nn = NearestNeighbors(
        n_neighbors=k,
        metric="cosine"
    )

    nn.fit(candidate_matrix)

    distances, indices = nn.kneighbors(query_matrix)

    # cosine distance = 1 - cosine similarity
    similarities = 1.0 - distances

    temperature = max(float(temperature), 1e-8)

    logits = similarities / temperature
    logits = logits - np.max(
        logits,
        axis=1,
        keepdims=True
    )

    weights = np.exp(logits)
    weights /= np.sum(
        weights,
        axis=1,
        keepdims=True
    )

    neighbor_targets = candidate_targets_log[indices]

    prediction_log = np.sum(
        weights * neighbor_targets,
        axis=1
    )

    return (
        prediction_log,
        indices,
        distances,
        weights
    )


(
    retrieval_pred_log,
    neighbor_indices,
    neighbor_distances,
    retrieval_weights
) = soft_retrieval_predict(
    X_test_final,
    X_train_final,
    y_train,
    RETRIEVAL_K,
    RETRIEVAL_TEMPERATURE
)

retrieval_pred_usd = np.maximum(
    np.expm1(retrieval_pred_log),
    0.0
)

print("Retrieval predictions generated.")


Retrieval predictions generated.



### Cell 18 — Explanation

This is the core retrieval operation.

For every test campaign:

1. Find the K nearest training campaigns.
2. Convert cosine distance into similarity.
3. Apply a temperature-scaled softmax.
4. Use those weights to average the retrieved campaigns' observed log funding.

\[
w_i=
\frac{e^{s_i/T}}
{\sum_j e^{s_j/T}}
\]

\[
\hat y_{retrieval}=
\sum_i w_i y_i
\]

So retrieval is essentially a **similar-campaign memory**.


In [23]:

# ============================================================
# 19. BLEND MLP + RETRIEVAL
# ============================================================

# alpha = 0.0 -> MLP only
# alpha = 1.0 -> retrieval only
# alpha = 0.3 -> 70% MLP + 30% retrieval

final_pred_log = (
    (1.0 - RETRIEVAL_ALPHA) * mlp_pred_log
    + RETRIEVAL_ALPHA * retrieval_pred_log
)

final_pred_usd = np.maximum(
    np.expm1(final_pred_log),
    0.0
)

print("MLP weight      :", 1.0 - RETRIEVAL_ALPHA)
print("Retrieval weight:", RETRIEVAL_ALPHA)


MLP weight      : 0.7
Retrieval weight: 0.3



### Cell 19 — Explanation

The final proposed model combines two sources of evidence:

**MLP:** learned global prediction function.

**Retrieval:** outcomes of similar historical campaigns.

\[
\hat y_{final}
=
(1-\alpha)\hat y_{MLP}
+
\alpha\hat y_{retrieval}
\]

This is the paper-inspired retrieval augmentation stage.


In [24]:

# ============================================================
# 20. EVALUATION
# ============================================================

def evaluate_predictions(
    model_name,
    y_true_log,
    y_pred_log,
    y_true_usd,
    y_pred_usd,
    training_time=None
):
    mse_log = mean_squared_error(
        y_true_log,
        y_pred_log
    )

    mse_usd = mean_squared_error(
        y_true_usd,
        y_pred_usd
    )

    result = {
        "Model": model_name,
        "MAE_log": mean_absolute_error(
            y_true_log,
            y_pred_log
        ),
        "MSE_log": mse_log,
        "RMSE_log": np.sqrt(mse_log),
        "R2_log": r2_score(
            y_true_log,
            y_pred_log
        ),
        "MAE_USD": mean_absolute_error(
            y_true_usd,
            y_pred_usd
        ),
        "MSE_USD": mse_usd,
        "RMSE_USD": np.sqrt(mse_usd),
        "R2_USD": r2_score(
            y_true_usd,
            y_pred_usd
        ),
        "RMSLE": np.sqrt(
            np.mean(
                (
                    np.log1p(np.maximum(y_true_usd, 0))
                    -
                    np.log1p(np.maximum(y_pred_usd, 0))
                ) ** 2
            )
        )
    }

    if training_time is not None:
        result["Training_Time_Seconds"] = training_time

    return result


xgb_metrics = evaluate_predictions(
    "Tuned XGBoost",
    y_test,
    xgb_pred_log,
    actual_usd,
    xgb_pred_usd
)

mlp_metrics = evaluate_predictions(
    "Tuned XGBoost Leaf Embedding + MLP",
    y_test,
    mlp_pred_log,
    actual_usd,
    mlp_pred_usd,
    mlp_training_time
)

method2_metrics = evaluate_predictions(
    "GBDT Embedding + MLP + Retrieval",
    y_test,
    final_pred_log,
    actual_usd,
    final_pred_usd,
    mlp_training_time
)

results_df = pd.DataFrame([
    xgb_metrics,
    mlp_metrics,
    method2_metrics
])

display(results_df.round(6))


,Model,MAE_log,MSE_log,RMSE_log,R2_log,MAE_USD,MSE_USD,RMSE_USD,R2_USD,RMSLE,Training_Time_Seconds
0,Tuned XGBoost,1.644995,4.999925,2.236051,0.467795,16040.364648,1.109358e+10,105326.062715,0.059985,2.236051,NaN
1,Tuned XGBoost Leaf Embedding + MLP,1.950523,6.353138,2.520543,0.323756,18051.906544,1.275300e+10,112929.176518,-0.080626,2.520543,11.263608
2,GBDT Embedding + MLP + Retrieval,1.845505,5.883581,2.425609,0.373737,16982.371058,1.165546e+10,107960.458482,0.012374,2.425609,11.263608



### Cell 20 — Explanation

This gives the three-way comparison:

1. **Tuned XGBoost**
2. **GBDT embedding + MLP**
3. **GBDT embedding + MLP + Retrieval**

The third row is the proposed Method 2.


In [25]:

# ============================================================
# 21. IMPROVEMENT CHECK
# ============================================================

baseline = results_df[
    results_df["Model"] == "Tuned XGBoost"
].iloc[0]

method2 = results_df[
    results_df["Model"] == "GBDT Embedding + MLP + Retrieval"
].iloc[0]

print("=" * 70)
print("METHOD 2 VS TUNED XGBOOST")
print("=" * 70)

print(
    "RMSE_log improvement:",
    baseline["RMSE_log"] - method2["RMSE_log"]
)

print(
    "R2_log improvement:",
    method2["R2_log"] - baseline["R2_log"]
)

print(
    "RMSE_USD improvement:",
    baseline["RMSE_USD"] - method2["RMSE_USD"]
)

print(
    "R2_USD improvement:",
    method2["R2_USD"] - baseline["R2_USD"]
)

if (
    method2["RMSE_log"] < baseline["RMSE_log"]
    and
    method2["R2_log"] > baseline["R2_log"]
):
    print("\nRESULT: Method 2 improved both RMSE_log and R2_log.")
else:
    print("\nRESULT: Method 2 did NOT beat the tuned XGBoost on both metrics.")


METHOD 2 VS TUNED XGBOOST
RMSE_log improvement: -0.1895581819401162
R2_log improvement: -0.09405856627055587
RMSE_USD improvement: -2634.3957663640904
R2_USD improvement: -0.04761101563808201

RESULT: Method 2 did NOT beat the tuned XGBoost on both metrics.



### Cell 21 — Explanation

This gives the direct answer to the research question.

For improvement:

- RMSE should go **down**.
- R² should go **up**.

The test set is evaluated only once here.


In [26]:

# ============================================================
# 22. RETRIEVAL INTERPRETABILITY
# ============================================================

INSPECT_N = min(5, len(X_test_final))

for q in range(INSPECT_N):
    print("=" * 70)
    print("TEST SAMPLE:", q)

    print("MLP log prediction      :", mlp_pred_log[q])
    print("Retrieval log prediction:", retrieval_pred_log[q])
    print("Final log prediction    :", final_pred_log[q])
    print("Actual log target       :", y_test[q])

    idx = neighbor_indices[q]
    dist = neighbor_distances[q]
    weight = retrieval_weights[q]

    table = pd.DataFrame({
        "train_index": idx,
        "cosine_distance": dist,
        "similarity": 1.0 - dist,
        "retrieval_weight": weight,
        "target_log": y_train[idx],
        "target_usd": np.maximum(
            np.expm1(y_train[idx]),
            0.0
        )
    })

    display(table.head(10))


TEST SAMPLE: 0
MLP log prediction      : 7.52209195169453
Retrieval log prediction: 6.834344297105904
Final log prediction    : 7.315767655317941
Actual log target       : 8.345948509766476


,train_index,cosine_distance,similarity,retrieval_weight,target_log,target_usd
0,9980,0.510972,0.489028,0.090511,8.431940,4590.398612
1,6968,0.542607,0.457393,0.065965,6.610533,741.879075
2,14685,0.553822,0.446178,0.058967,3.044522,20.000000
3,12598,0.563871,0.436129,0.053329,6.839476,933.000000
4,13874,0.569865,0.430135,0.050227,8.755580,6345.000000
5,15525,0.574497,0.425503,0.047953,2.962137,18.339261
6,9595,0.584688,0.415312,0.043307,8.606669,5467.002172
7,6002,0.593918,0.406082,0.039489,8.757368,6356.356436
8,15619,0.595040,0.404960,0.039048,4.386556,79.363181
9,5148,0.603808,0.396192,0.035770,7.334854,1531.803936


TEST SAMPLE: 1
MLP log prediction      : 6.107825037153186
Retrieval log prediction: 5.893238419493013
Final log prediction    : 6.043449051855134
Actual log target       : 7.601402334583733


,train_index,cosine_distance,similarity,retrieval_weight,target_log,target_usd
0,4353,0.539256,0.460744,0.089474,5.398163,220.000000
1,11359,0.564180,0.435820,0.069736,0.000000,0.000000
2,1498,0.573700,0.426300,0.063403,8.285513,3965.000000
3,7359,0.596528,0.403472,0.050462,7.523541,1850.110000
4,5183,0.607891,0.392109,0.045042,7.095064,1205.000000
5,12959,0.612400,0.387600,0.043056,0.000000,0.000000
6,14252,0.625491,0.374509,0.037773,7.881182,2646.000000
7,15503,0.632917,0.367083,0.035070,4.653801,103.983228
8,1424,0.640489,0.359511,0.032512,7.376252,1596.590000
9,2236,0.645647,0.354353,0.030878,8.415547,4515.745097


TEST SAMPLE: 2
MLP log prediction      : 8.976444972414807
Retrieval log prediction: 8.357118048260036
Final log prediction    : 8.790646895168376
Actual log target       : 8.842687992718748


,train_index,cosine_distance,similarity,retrieval_weight,target_log,target_usd
0,12829,0.584110,0.415890,0.060895,9.494097,13280.100884
1,9948,0.595190,0.404810,0.054508,4.909265,134.539789
2,13650,0.606328,0.393672,0.048763,9.493186,13268.000000
3,4818,0.607013,0.392987,0.048430,9.741575,17009.308943
4,2619,0.608650,0.391350,0.047643,6.776507,876.000000
5,1589,0.614666,0.385334,0.044862,9.102644,8978.000000
6,4495,0.619572,0.380428,0.042714,8.423821,4553.271395
7,14489,0.633839,0.366161,0.037034,5.666427,288.000000
8,15587,0.641426,0.358574,0.034329,9.084891,8820.000000
9,10460,0.642357,0.357643,0.034011,6.688355,802.000000


TEST SAMPLE: 3
MLP log prediction      : 2.6146792462185187
Retrieval log prediction: 4.9924858773319
Final log prediction    : 3.3280212355525327
Actual log target       : 0.6931471805599453


,train_index,cosine_distance,similarity,retrieval_weight,target_log,target_usd
0,8963,0.431534,0.568466,0.220236,3.119914,21.644428
1,15373,0.541607,0.458393,0.073257,9.482731,13130.000000
2,1505,0.554476,0.445524,0.064411,4.276666,71.000000
3,2373,0.575147,0.424853,0.052383,2.833213,16.000000
4,15784,0.592562,0.407438,0.044010,5.356539,210.990000
5,3623,0.607306,0.392694,0.037977,4.624973,101.000000
6,15269,0.624125,0.375875,0.032098,5.220356,184.000000
7,13537,0.625634,0.374366,0.031617,0.000000,0.000000
8,5322,0.635830,0.364170,0.028552,5.632183,278.271127
9,11824,0.649457,0.350543,0.024915,4.624973,101.000000


TEST SAMPLE: 4
MLP log prediction      : 7.571267871661439
Retrieval log prediction: 7.915970004277563
Final log prediction    : 7.674678511446276
Actual log target       : 5.749575820307675


,train_index,cosine_distance,similarity,retrieval_weight,target_log,target_usd
0,10848,0.526128,0.473872,0.093938,8.047909,3126.248279
1,8775,0.557440,0.442560,0.068684,10.421031,33557.007949
2,12735,0.589942,0.410058,0.049625,10.221105,27476.000000
3,5729,0.594563,0.405437,0.047384,6.670766,788.000000
4,9464,0.596115,0.403885,0.046654,6.216606,500.000000
5,5105,0.597946,0.402054,0.045808,3.713572,40.000000
6,13122,0.620503,0.379497,0.036558,9.810714,18227.000000
7,11587,0.623405,0.376595,0.035512,9.211266,10008.259905
8,6549,0.626323,0.373677,0.034491,1.356695,2.883339
9,2234,0.629479,0.370521,0.033419,10.622382,41042.247590



### Cell 22 — Explanation

This is the main interpretability advantage of retrieval.

For a new campaign, you can show:

**new campaign → similar historical campaigns → their funding outcomes → retrieval prediction**

This makes the retrieval component easier to explain in a thesis or presentation.


In [27]:

# ============================================================
# 23. RETRIEVAL DIAGNOSTICS
# ============================================================

top1_similarity = 1.0 - neighbor_distances[:, 0]

print("Top-1 similarity:")
print(pd.Series(top1_similarity).describe())

print(
    "\nAverage maximum retrieval weight:",
    np.mean(np.max(retrieval_weights, axis=1))
)

print(
    "\nAverage top-5 similarity:",
    np.mean(1.0 - neighbor_distances[:, :5])
)


Top-1 similarity:
count    4000.000000
mean        0.490280
std         0.070217
min         0.337633
25%         0.439963
50%         0.478787
75%         0.528501
max         0.980923
dtype: float64

Average maximum retrieval weight: 0.11359035655938927

Average top-5 similarity: 0.4378592983501975



### Cell 23 — Explanation

If retrieval is useful, the nearest campaigns should generally have meaningful
similarity.

These diagnostics help determine whether retrieval is actually finding useful
historical neighbors or simply adding noise.


In [28]:

# ============================================================
# 24. SAVE ALL MODEL COMPONENTS
# ============================================================

joblib.dump(
    xgb_model,
    OUTPUT_DIR / "tuned_xgb_model.pkl"
)

joblib.dump(
    robust_preprocessor,
    OUTPUT_DIR / "robust_preprocessor.pkl"
)

joblib.dump(
    leaf_encoder,
    OUTPUT_DIR / "leaf_encoder.pkl"
)

joblib.dump(
    Omega,
    OUTPUT_DIR / "random_projection.pkl"
)

joblib.dump(
    pca,
    OUTPUT_DIR / "pca.pkl"
)

joblib.dump(
    final_scaler,
    OUTPUT_DIR / "final_scaler.pkl"
)

joblib.dump(
    mlp,
    OUTPUT_DIR / "mlp_model.pkl"
)

joblib.dump(
    retrieval_index,
    OUTPUT_DIR / "retrieval_index.pkl"
)

joblib.dump(
    X_train_final,
    OUTPUT_DIR / "retrieval_candidate_embeddings.pkl"
)

joblib.dump(
    y_train,
    OUTPUT_DIR / "retrieval_candidate_targets_log.pkl"
)

joblib.dump(
    {
        "retrieval_k": RETRIEVAL_K,
        "retrieval_temperature": RETRIEVAL_TEMPERATURE,
        "retrieval_alpha": RETRIEVAL_ALPHA,
        "retrieval_metric": "cosine",
        "random_dim": RANDOM_DIM,
        "main_dim": effective_main_dim,
        "seed": SEED
    },
    OUTPUT_DIR / "retrieval_config.pkl"
)

print("All model components saved.")


All model components saved.



### Cell 24 — Explanation

This saves everything required for later inference.

Most importantly, the retrieval memory consists of:

- the fitted nearest-neighbor index,
- training embeddings,
- training target values.

The test targets are never stored in the retrieval memory.


In [29]:

# ============================================================
# 25. SAVE METRICS + PREDICTIONS + METADATA
# ============================================================

results_df.to_csv(
    OUTPUT_DIR / "method2_metrics.csv",
    index=False
)

prediction_df = pd.DataFrame({
    "actual_usd": actual_usd,
    "actual_log": y_test,

    "xgb_pred_log": xgb_pred_log,
    "xgb_pred_usd": xgb_pred_usd,

    "mlp_pred_log": mlp_pred_log,
    "mlp_pred_usd": mlp_pred_usd,

    "retrieval_pred_log": retrieval_pred_log,
    "retrieval_pred_usd": retrieval_pred_usd,

    "final_pred_log": final_pred_log,
    "final_pred_usd": final_pred_usd,

    "top1_retrieval_similarity": top1_similarity
})

prediction_df.to_csv(
    OUTPUT_DIR / "method2_predictions.csv",
    index=False
)

metadata = {
    "method": "GBDT leaf embedding + MLP + retrieval",
    "xgb_model": "previously fitted tuned XGBoost",
    "target": TARGET,
    "log_target": LOG_TARGET,
    "n_train": int(len(train)),
    "n_test": int(len(test)),
    "n_features": int(len(feature_cols)),
    "random_dim": int(RANDOM_DIM),
    "pca_dim": int(effective_main_dim),
    "retrieval_k": int(RETRIEVAL_K),
    "retrieval_temperature": float(RETRIEVAL_TEMPERATURE),
    "retrieval_alpha": float(RETRIEVAL_ALPHA),
    "retrieval_metric": "cosine",
    "seed": SEED
}

with open(
    OUTPUT_DIR / "method2_metadata.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, indent=2)

print("Saved:")
print("- method2_metrics.csv")
print("- method2_predictions.csv")
print("- method2_metadata.json")


Saved:
- method2_metrics.csv
- method2_predictions.csv
- method2_metadata.json



### Cell 25 — Explanation

Saves:

- all metrics,
- every prediction,
- retrieval similarity,
- configuration and experiment metadata.

This makes the experiment reproducible and easy to compare with your baseline tables.


In [30]:

# ============================================================
# 26. FINAL SUMMARY
# ============================================================

display(
    results_df[
        [
            "Model",
            "MAE_log",
            "RMSE_log",
            "R2_log",
            "MAE_USD",
            "RMSE_USD",
            "R2_USD",
            "RMSLE"
        ]
    ].round(6)
)

print("\nOutput directory:")
print(OUTPUT_DIR)


,Model,MAE_log,RMSE_log,R2_log,MAE_USD,RMSE_USD,R2_USD,RMSLE
0,Tuned XGBoost,1.644995,2.236051,0.467795,16040.364648,105326.062715,0.059985,2.236051
1,Tuned XGBoost Leaf Embedding + MLP,1.950523,2.520543,0.323756,18051.906544,112929.176518,-0.080626,2.520543
2,GBDT Embedding + MLP + Retrieval,1.845505,2.425609,0.373737,16982.371058,107960.458482,0.012374,2.425609



Output directory:
E:\NSU\cse445\EDA attempt3\ML models\GBDT pipeline\GBDT_prop_2



### Cell 26 — Explanation

This is the final result table you should use for your experiment report.

The critical comparison is:

**Tuned XGBoost vs GBDT Embedding + MLP + Retrieval.**
